# Extração de dados

Este notebook corresponde à etapa de extração das fontes brutas usadas no trabalho.


In [1]:
import json
import os
import glob
from pathlib import Path
import re
import logging
import multiprocessing as mp
from datetime import datetime,timezone
from functools import partial
import pandas as pd
import polars as pl
from tqdm import tqdm
import ast
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures import ThreadPoolExecutor, as_completed

## 0.1 Explorar um arquivo JSON

### 0.1.1 Estrutura dos arquivos

In [ ]:
categorias = ["beauty","fashion","fitness","food","travel","family","interior","pet","fitness","other"]

sample_path = Path("Diretório do conjunto de dados original")

extensoes = {}
for f in sample_path.rglob("*"):          # rglob = glob recursivo nativo do pathlib
    if f.is_file():
        ext = f.suffix.lower()     # .json, .info, .txt, etc.
        extensoes[ext] = extensoes.get(ext, 0) + 1

In [8]:
print("Extensões encontradas:")
for ext, qtd in sorted(extensoes.items(), key=lambda x: -x[1]):
    print(f"  {ext or '(sem extensão)'}: {qtd:,} arquivos")

Extensões encontradas:
  .info: 9,726,219 arquivos


In [9]:
print("\nEstrutura (5 níveis):")
for i, item in enumerate(sample_path.rglob("*")):
    print(" ", item.relative_to(sample_path))
    if i > 20:
        print("  ...")
        break


Estrutura (5 níveis):
  00s_supermodels-1845862301087188249.info
  00s_supermodels-1845877547776500327.info
  00s_supermodels-1845881622442155757.info
  00s_supermodels-1846300638369512714.info
  00s_supermodels-1847373891523510539.info
  00s_supermodels-1847920323502399071.info
  00s_supermodels-1847922129401919823.info
  00s_supermodels-1848076291783557681.info
  00s_supermodels-1848112080949186303.info
  00s_supermodels-1848762098521553998.info
  00s_supermodels-1850697318728486967.info
  00s_supermodels-1850945237444494103.info
  00s_supermodels-1850999461415187263.info
  00s_supermodels-1851337950396665003.info
  00s_supermodels-1851448973405305365.info
  00s_supermodels-1851760682443634289.info
  00s_supermodels-1851766773378404834.info
  00s_supermodels-1852101659494613502.info
  00s_supermodels-1852470026994934114.info
  00s_supermodels-1853070162548573727.info
  00s_supermodels-1853625301894189029.info
  00s_supermodels-1853840120018783238.info
  ...


### 0.1.2 Inspecionar um arquivo .info

In [ ]:
base = Path("Diretório do conjunto de dados original")
primeiro = next(base.rglob("*.info"),None)

In [12]:
if primeiro is None:
    print("Nenhum .info encontrado. Listando primeiros arquivos:")
    for f in list(base.rglob("*"))[:10]:
        print(" ", f)
else:
    print(f"Arquivo encontrado: {primeiro}")
    print(f"Tamanho: {primeiro.stat().st_size} bytes\n")
    
    with open(primeiro, "r", encoding="utf-8") as f:
        conteudo = f.read(2000)   # primeiros 2000 chars
    print(conteudo)

Arquivo encontrado: C:\Users\cruzd\Downloads\Instagram Influencer DataSet\Post_metadata\info\00s_supermodels-1845862301087188249.info
Tamanho: 8879 bytes

{"gating_info": null, "viewer_can_reshare": true, "display_resources": [{"src": "https://scontent-lax3-1.cdninstagram.com/vp/5b0d4aa05dd4f0445371f33cad1ffcf8/5DD75103/t51.2885-15/sh0.08/e35/p640x640/38097578_1847504855364056_3032458266316636160_n.jpg?_nc_ht=scontent-lax3-1.cdninstagram.com", "config_width": 640, "config_height": 800}, {"src": "https://scontent-lax3-1.cdninstagram.com/vp/d99a6dd7829e8d7f803ebc02b5cb7bfc/5DE1EEC7/t51.2885-15/sh0.08/e35/p750x750/38097578_1847504855364056_3032458266316636160_n.jpg?_nc_ht=scontent-lax3-1.cdninstagram.com", "config_width": 750, "config_height": 937}, {"src": "https://scontent-lax3-1.cdninstagram.com/vp/abdf18797f8b85a8e5816c72d91335ca/5DC8EE69/t51.2885-15/e35/38097578_1847504855364056_3032458266316636160_n.jpg?_nc_ht=scontent-lax3-1.cdninstagram.com", "config_width": 1080, "config_height":

### 0.1.3 Visualização de um arquivo

In [13]:
arquivos = list(base.rglob("*.info"))
print(f"Total de arquivos: {len(arquivos)}")
print(f"Exemplo: {arquivos[0]}")

Total de arquivos: 9726219
Exemplo: C:\Users\cruzd\Downloads\Instagram Influencer DataSet\Post_metadata\info\00s_supermodels-1845862301087188249.info


In [16]:
with open(arquivos[0],"r",encoding="utf-8") as f:
    post = json.load(f)

print(f"Chaves: {list(post.keys())}")
print("\nConteudo:")
for k,v in post.items():
    print(f"{k}: {repr(str(v))}")

Chaves: ['gating_info', 'viewer_can_reshare', 'display_resources', 'viewer_in_photo_of_you', 'viewer_has_saved_to_collection', 'viewer_has_saved', 'owner', 'viewer_has_liked', 'id', 'should_log_client_event', 'edge_media_preview_like', 'edge_media_to_tagged_user', 'dimensions', '__typename', 'location', 'shortcode', 'is_ad', 'caption_is_edited', 'media_preview', 'edge_sidecar_to_children', 'taken_at_timestamp', 'edge_media_to_caption', 'tracking_token', 'has_ranked_comments', 'display_url', 'edge_media_to_comment', 'comments_disabled', 'edge_media_to_sponsor_user', 'is_video', 'edge_web_media_to_related_media']

Conteudo:
gating_info: 'None'
viewer_can_reshare: 'True'
display_resources: "[{'src': 'https://scontent-lax3-1.cdninstagram.com/vp/5b0d4aa05dd4f0445371f33cad1ffcf8/5DD75103/t51.2885-15/sh0.08/e35/p640x640/38097578_1847504855364056_3032458266316636160_n.jpg?_nc_ht=scontent-lax3-1.cdninstagram.com', 'config_width': 640, 'config_height': 800}, {'src': 'https://scontent-lax3-1.cdni

### 0.1.4 Schema real mapeado

posts .info<br>
│<br>
├── IDENTIFICAÇÃO<br>
│   ├── shortcode              -> 'Bmd0fuolkUZ'<br>
│   ├── id                     -> '1845862301087188249'<br>
│   └── taken_at_timestamp     -> '1534263955' <br>
│<br>
├── INFLUENCIADOR (owner)<br>
│   ├── username               -> '00s_supermodels'<br>
│   ├── id                     -> '3007149350'<br>
│   ├── is_verified            -> False<br>
│   ├── is_private             -> False<br>
│<br>
├── ENGAJAMENTO<br>
│   ├── edge_media_preview_like['count']    -> 66 (likes)<br>
│   └── edge_media_to_comment['count']      -> 4  (total de comentários)<br>
│<br>
├── COMENTÁRIOS (edge_media_to_comment['edges'])<br>
│   └── Cada comentário tem:<br>
│       ├── text               -> 'Like it 😎 If you had three wishes...'<br>
│       ├── created_at         -> 1534264510 (Unix, INT nativo)<br>
│       ├── edge_liked_by count-> 0  (likes no comentário)<br>
│       └── owner.username     -> 'veganundautark'<br>
│       Limitado a ~10 comentários (preview da API)<br>
│<br>
├── CAPTION<br>
│   └── edge_media_to_caption['edges'][0]['node']['text']<br>
│       -> '#AdCampaign\n(Scan By Me)\n#AnastassiaKhozissova...'<br>
│<br>
├── PATROCÍNIO — 3 SINAIS<br>
│   ├── is_ad                  -> 'False' <br>
│   ├── edge_media_to_sponsor_user['edges'] -> [] (vazio)<br>
│   └── Caption contém: #AdCampaign #Advertisement #Advertising #ads<br>
│<br>
├── TIPO DE POST<br>
│   ├── is_video               -> 'False'<br>
│   ├── __typename             -> 'GraphSidecar' (= carousel)<br>
│   └── edge_sidecar_to_children['edges'] -> 2 imagens<br>
│<br>
├── IMAGEM (dentro de edge_sidecar_to_children)<br>
│   └── accessibility_caption  -> 'Image may contain: 2 people, indoor' <br>
│       (análise de visão computacional do próprio Instagram!)<br>
│<br>
└── CONTEÚDO EXTRA<br>
    ├── dimensions             -> {'width': 1080, 'height': 1350}<br>
    ├── location               -> None<br>
    └── edge_media_to_tagged_user -> 1 usuário marcado<br>
<br>
<br>
influencers.txt<br>
│<br>
├── IDENTIFICAÇÃO<br>
│   ├── username       -> 'makeupbynvs'<br>
│   ├── category       -> 'beauty'<br>
│   ├── #Followers     -> '1432' <br>
│   ├── #Followees     -> '1089' <br>
│   └── #Posts         -> '363' <br>

# 1.0 Extração dos dados


## 1.1 Configurações

In [ ]:
BASE = Path("Diretório do conjunto de dados original")
SAIDA = Path("Diretorio de saida dos baches")
SAIDA.mkdir(exist_ok=True)
TXT = Path("Caminho do arquivo que mapeia influenciadores")

BATCH_SIZE = 100_000
N_WORKERS = max(1,mp.cpu_count()-2)

In [3]:
logging.basicConfig(level=logging.INFO,format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

## 1.2 Extração de influenciadores

O arquivo influencers.txt tem colunas separadas por TAB: Username	Category	#Followers	#Followees	#Posts

Vamos tranformar esse TXT em CSV com os dados dos influenciadores

In [4]:
linhas = []
with open(TXT,"r",encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha or linha.startswith("="):
            continue
        linhas.append(linha.split("\t"))     

In [5]:
header = [c.strip().lstrip("#") for c in linhas[0]] # Primeira linha é o header

df_inf = pd.DataFrame(linhas[1:],columns=header)

In [6]:
df_inf.info()

<class 'pandas.DataFrame'>
RangeIndex: 33935 entries, 0 to 33934
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Username   33935 non-null  str  
 1   Category   33935 non-null  str  
 2   Followers  33935 non-null  str  
 3   Followees  33935 non-null  str  
 4   Posts      33935 non-null  str  
dtypes: str(5)
memory usage: 1.3 MB


In [7]:
# Dados em dtype string tem que ser numeric
df_inf["Followers"] = pd.to_numeric(df_inf['Followers'],errors="coerce").fillna(0).astype(int)
df_inf["Followees"] = pd.to_numeric(df_inf['Followees'],errors="coerce").fillna(0).astype(int)
df_inf["Posts"] = pd.to_numeric(df_inf['Posts'],errors="coerce").fillna(0).astype(int)

In [8]:
df_inf.info()

<class 'pandas.DataFrame'>
RangeIndex: 33935 entries, 0 to 33934
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Username   33935 non-null  str  
 1   Category   33935 non-null  str  
 2   Followers  33935 non-null  int64
 3   Followees  33935 non-null  int64
 4   Posts      33935 non-null  int64
dtypes: int64(3), str(2)
memory usage: 1.3 MB


In [9]:
df_inf.groupby("Category")["Followers"].agg(["count","median","max"]).round(4)

,count,median,max
Category,,,
beauty,1542,18249.0,13957201
family,4070,11188.0,80465206
fashion,11911,23560.0,96476007
fashion 0.5,1,721857.0,721857
fasion,1,213515.0,213515
fitness,1133,28290.0,28834289
food,3565,12554.0,2730067
interior,1195,19957.0,1703450
other,5720,8718.0,80852821


In [10]:
# Corrigindo o erro de categoria fasion e fashion 0.5
CORRECOES = {
    "fasion":      "fashion",
    "fashion 0.5": "fashion",
}
 
df_inf["Category"] = df_inf["Category"].replace(CORRECOES)

print(sorted(df_inf["Category"].unique()))

['beauty', 'family', 'fashion', 'fitness', 'food', 'interior', 'other', 'pet', 'travel']


### 1.2.1 Salvando o df_inf (dataframe de influenciadores) localmente

In [11]:
df_inf.to_csv("dataframe_influencers.csv",index=False)

## 1.3 Funções auxiliares

### 1.3.1 Conversões de tipo e tratamento de exceções

In [ ]:
def para_bool(v) -> bool:
    """"'True' e 'False' em string nos arquivos json, são convertidos para bool"""
    if isinstance(v,bool):
        return v
    return str(v).strip().lower() in {"true","1","yes","y"}

def para_int(v,default: int = 0) -> int:
    """Qualquer valor para int, ou 'default' se falhar"""
    try:
        return int(v)
    except(TypeError,ValueError):
        return default
    
def ts_para_data(ts) -> str|None:
    """Timestamp unix para yyyy-mm-dd"""
    try:
        return datetime.fromtimestamp(int(ts),tz=timezone.utc).strftime("%Y-%m-%d")
    except(TypeError,ValueError,OSError):
        return None


### 1.3.2 Extração de campos específicos

In [13]:
def extrair_caption(p: dict) -> str:
    """
    Extrai o texto da legenda (caption) do post.
    Estrutura no arquivo: edge_media_to_caption → edges → [0] → node → text
    """
    try:
        return p["edge_media_to_caption"]["edges"][0]["node"]["text"] or ""
    except (KeyError, IndexError, TypeError):
        return ""
 
 
def extrair_hashtags(texto: str) -> list[str]:
    """
    Encontra todas as hashtags no texto usando regex.
    '#AdCampaign' > ['adcampaign']
    O .lower() padroniza para comparação.
    """
    return re.findall(r"#(\w+)", texto.lower())

In [14]:
def tipo_post(p:dict)->str:
    """
    Classifica o tipo do post:
    - 'video'    > is_video = True
    - 'carousel' > __typename = 'GraphSidecar'
    - 'image'    > caso contrário
    """
    if para_bool(p.get("is_video", False)):
        return "video"
    if p.get("__typename") == "GraphSidecar":
        return "carousel"
    return "image"

In [15]:
def extrair_accessibility(p: dict) -> str:
    """
    Concatena as legendas de acessibilidade de todas as imagens
    do carrossel, separadas por ' | '.
    """
    try:
        sidecar = p.get("edge_sidecar_to_children", {})
        if not isinstance(sidecar, dict):
            return ""
        edges = sidecar.get("edges", [])
        partes = [
            edge["node"].get("accessibility_caption", "") or ""
            for edge in edges
            if edge.get("node")
        ]
        return " | ".join(p for p in partes if p)
    except (KeyError, TypeError):
        return ""
 

In [16]:
def aspect_ratio(p:dict)->float|None:
    """Calcula largura/altura da imagem principal do post"""
    try:
        d = p.get("dimensions", {})
        return round(d["width"] / d["height"], 3)
    except (KeyError, TypeError, ZeroDivisionError):
        return None

#### 1.3.3 Verificar hashtags de publi/ads

In [17]:
TAGS_PUBLI = {"ad", "ads", "sponsored", "sponsorship", "parceria", "publi",
              "publicidade", "patrocinado", "adcampaign", "brandedcontent",
              "advertisement", "advertising", "collaboration", "collab", "gifted",
              }

In [18]:
def detectar_sponsored(p:dict,caption:str)->bool:
    """
    Detecta se o post é patrocinado verificando: 
        Campo is_ad = True,
        Tem usuário patrocinador em edge_media_to_sponsor_user,
        Caption contém hashtags de publicidade (#ad, #publi, etc.).
    """
    if para_bool(p.get("is_ad", False)):
        return True
    sponsor = p.get("edge_media_to_sponsor_user", {})
    if isinstance(sponsor, dict) and sponsor.get("edges"):
        return True
    if set(extrair_hashtags(caption)) & TAGS_PUBLI:
        return True
    return False

## 1.4 Construir lookup a partir do df_inf
Dicionario feito com o dataframe de influenciadores

In [19]:
df_inf = pd.read_csv("dataframe_influencers.csv")

In [20]:
lookup = (df_inf.set_index("Username")[["Category", "Followers", "Followees", "Posts"]]
    .rename(columns={
        "Category":  "inf_category",
        "Followers": "followers",
        "Followees": "followees",
        "Posts":     "posts_total",
    })
    .to_dict("index")
)
 
print(f"Lookup pronto: {len(lookup):,} influenciadores")

Lookup pronto: 33,935 influenciadores


## 1.5 Extração de arquivo

In [21]:
def extrair_post_e_comentarios(args:tuple) -> tuple[dict|None,list[dict]]:
    """
    Esta função recebe uma TUPLA (filepath, lookup) e retorna:
        - post_row:      dict com todas as métricas do post
        - comment_rows:  lista de dicts, um por comentário
    É chamada em paralelo por N_WORKERS processos ao mesmo tempo.
    """

    filepath,lookup = args

    #leitura do arquivo
    try: 
        with open(filepath,"r",encoding="utf-8",errors="ignore") as f:
            p = json.load(f)
    except Exception:
        # descarta silenciosamente se o arquivo for corrompido ou ilegível 
        return None, []
    
    #Identificação do username e post_id
    stem = filepath.stem
    nome_partes = stem.rsplit("-",1)
    username_fallback = nome_partes[0] if len(nome_partes) == 2 else stem

    owner = p.get("owner",{})
    username = owner.get("username",username_fallback) or username_fallback
    post_id = str(p.get("shortcode",filepath.stem))

    # Dados do influenciador via lookup
    inf_data = lookup.get(username, {})
    followers    = inf_data.get("followers",   0)
    followees    = inf_data.get("followees",   0)
    posts_total  = inf_data.get("posts_total", 0)
    inf_category = inf_data.get("inf_category", "")

    #Timestamp
    ts = p.get("taken_at_timestamp")

    #Likes e comments
    likes_data = p.get("edge_media_preview_like",{})
    comments_data = p.get("edge_media_to_comment",  {})
    likes         = para_int(likes_data.get("count", 0))
    comments_cnt  = para_int(comments_data.get("count", 0))

    #Caption e hashtags
    caption  = extrair_caption(p)
    hashtags = extrair_hashtags(caption)
 
    #Número de imagens do carrossel
    sidecar   = p.get("edge_sidecar_to_children", {})
    n_imagens = max(len(sidecar.get("edges", [])), 1)

    #Usuarios marcados no post
    tagged    = p.get("edge_media_to_tagged_user", {})
    n_usertags = len(tagged.get("edges", []))

    # Monta o registo do post
    post_row = {
        # Identificação
        "post_id":        post_id,
        "username":       username,
        "inf_category":   inf_category,
 
        # Influenciador
        "followers":      followers,
        "followees":      followees,
        "posts_total":    posts_total,
        "is_verified":    para_bool(owner.get("is_verified", False)),
 
        # Temporal
        "data":           ts_para_data(ts),
        "timestamp_unix": para_int(ts),
 
        # Engajamento bruto (ER calculado depois, na etapa de preparação)
        "likes":          likes,
        "comments_count": comments_cnt,
 
        # Tipo e formato
        "post_type":      tipo_post(p),
        "n_imagens":      n_imagens,
        "aspect_ratio":   aspect_ratio(p),
 
        # Conteúdo
        "is_sponsored":   detectar_sponsored(p, caption),
        "n_hashtags":     len(hashtags),
        "n_usertags":     n_usertags,
        "caption_len":    len(caption),
        "has_location":   p.get("location") not in [None, "None", ""],
        "accessibility":  extrair_accessibility(p),
    }

    # Monta o registro dos comentarios
    comment_rows = []
    for edge in comments_data.get("edges", []):
        node  = edge.get("node", {})
        texto = node.get("text", "")
        if not texto:
            continue
        liked_by  = node.get("edge_liked_by", {})
        owner_c   = node.get("owner", {})
        comment_rows.append({
            "post_id":            post_id,
            "username":           username,
            "inf_category":       inf_category,
            "comment_id":         str(node.get("id", "")),
            "texto":              texto,
            "created_at":         ts_para_data(node.get("created_at")),
            "likes_comentario":   para_int(liked_by.get("count", 0)),
            "commenter":          owner_c.get("username"),
            "commenter_verified": para_bool(owner_c.get("is_verified", False)),
        })
 
    return post_row, comment_rows

# 2.0 Loop Principal com multiprocessing


In [22]:
arquivos = list(BASE.rglob("*.info"))
print(f"Total de Arquivos: {len(arquivos)}")

Total de Arquivos: 9726219


In [ ]:
def chunks(lst, n):
    """Divide uma lista em pedaços de tamanho n."""
    for i in range(0, len(lst), n):
        yield lst[i : i + n]

In [24]:
CHUNK_SUBMIT = 5_000

posts_b, comments_b = [], []
batch_num, erros = 0, 0


with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    with tqdm(total=len(arquivos), desc="Extraindo") as barra:
        for chunk in chunks(arquivos, CHUNK_SUBMIT):
            futuros = {executor.submit(extrair_post_e_comentarios, (f, lookup)): f for f in chunk}

            for futuro in as_completed(futuros):
                try:
                    post_row, comment_rows = futuro.result()
                except Exception as e:
                    log.warning(f"Erro: {e}")
                    erros += 1
                    barra.update(1)
                    continue

                if post_row is None:
                    erros += 1
                    barra.update(1)
                    continue

                posts_b.append(post_row)
                comments_b.extend(comment_rows)
                barra.update(1)

                if len(posts_b) >= BATCH_SIZE:
                    pl.DataFrame(posts_b).write_parquet(
                        SAIDA / f"posts_batch_{batch_num:04d}.parquet",
                        compression="zstd",
                    )
                    if comments_b:
                        pl.DataFrame(comments_b).write_parquet(
                            SAIDA / f"comments_batch_{batch_num:04d}.parquet",
                            compression="zstd",
                        )
                    posts_b, comments_b = [], []
                    batch_num += 1

if posts_b:
    pl.DataFrame(posts_b).write_parquet(
        SAIDA / f"posts_batch_{batch_num:04d}.parquet",
        compression="zstd",
    )
if comments_b:
    pl.DataFrame(comments_b).write_parquet(
        SAIDA / f"comments_batch_{batch_num:04d}.parquet",
        compression="zstd",
    )

print(f"\nBatches salvos: {batch_num + 1}")
print(f"Erros ignorados: {erros:,}")

Extraindo: 100%|██████████| 9726219/9726219 [3:53:42<00:00, 693.59it/s]   



Batches salvos: 98
Erros ignorados: 21


# 3.0 Combinando batches e salvando parquet final

In [25]:
log.info("Combinando posts...")
(
    pl.scan_parquet(str(SAIDA / "posts_batch_*.parquet"))
    .sink_parquet("posts.parquet", compression="zstd")
)
 
log.info("Combinando comentários...")
(
    pl.scan_parquet(str(SAIDA / "comments_batch_*.parquet"))
    .sink_parquet("comments.parquet", compression="zstd")
)
 
# Leitura só para exibir estatísticas finais
df_posts    = pl.read_parquet("posts.parquet")
df_comments = pl.read_parquet("comments.parquet")

2026-04-18 23:02:57,131 INFO Combinando posts...
2026-04-18 23:02:58,794 INFO Combinando comentários...


In [26]:
print(f"\nposts.parquet:    {df_posts.shape[0]:>12,} linhas x {df_posts.shape[1]} colunas")
print(f"comments.parquet: {df_comments.shape[0]:>12,} linhas x {df_comments.shape[1]} colunas")
print(f"Erros ignorados:  {erros:>12,}")


posts.parquet:       9,726,198 linhas x 20 colunas
comments.parquet:   18,135,110 linhas x 9 colunas
Erros ignorados:            21


In [27]:
df_posts.head()

post_id,username,inf_category,followers,followees,posts_total,is_verified,data,timestamp_unix,likes,comments_count,post_type,n_imagens,aspect_ratio,is_sponsored,n_hashtags,n_usertags,caption_len,has_location,accessibility
str,str,str,i64,i64,i64,bool,str,i64,i64,i64,str,i64,f64,bool,i64,i64,i64,bool,str
"""BmlIb5Xld5f""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-17""",1534509290,51,2,"""image""",1,0.8,false,18,1,257,false,""""""
"""BmlI2LPlblP""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-17""",1534509505,89,0,"""image""",1,0.8,false,17,1,279,false,""""""
"""BmoH1VclTxO""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-18""",1534609638,94,0,"""image""",1,0.8,false,16,1,228,false,""""""
"""Bmd0fuolkUZ""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-14""",1534263955,66,4,"""carousel""",2,0.8,true,17,1,282,false,"""Image may contain: 2 people, i…"
"""Bmd445DFh7t""","""00s_supermodels""","""fashion""",2031,53,2520,false,"""2018-08-14""",1534266258,64,0,"""carousel""",4,0.8,false,18,1,255,false,"""Image may contain: 1 person, c…"
